In [2]:
'''
from google.colab import drive
drive.mount('/content/drive')
'''

"\nfrom google.colab import drive\ndrive.mount('/content/drive')\n"

In [9]:
#pip uninstall -y opencv-contrib-python opencv-python opencv-python-headless mediapipe numpy protobuf

In [10]:
"""%pip install numpy==1.26.4
%pip install protobuf==4.25.3
%pip install opencv-python-headless==4.9.0.80
%pip install mediapipe==0.10.32 --no-deps"""

'%pip install numpy==1.26.4\n%pip install protobuf==4.25.3\n%pip install opencv-python-headless==4.9.0.80\n%pip install mediapipe==0.10.32 --no-deps'

In [3]:
# ==========================================================
#   FINAL — TRAIN LOCAL HIERARCHICAL MODELS (WORKING)
# ==========================================================
import os, joblib, ast
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks



In [4]:
import mediapipe as mp
import numpy as np
import cv2

mp_hands = mp.solutions.hands

hands_detector = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def frame_to_vector(frame):
    if frame is None:
        return None

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = hands_detector.process(rgb)

    if not res.multi_hand_landmarks:
        return None

    lm = res.multi_hand_landmarks[0]

    vec = []
    for p in lm.landmark:
        vec.extend([p.x, p.y, p.z])

    return np.array(vec, dtype="float32").reshape(1, 63)

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [9]:


# ================= PATHS =============================
DATA_CSV = "/content/drive/MyDrive/handshape_combinations_features_updated.csv"
OUT_DIR = "/content/drive/MyDrive/hier_sravs_models"
os.makedirs(OUT_DIR, exist_ok=True)

# ================= NODE DEFINITIONS ==================
NODES = {
    "A": [
        "hamfist","hamflathand",
        "hamfinger2","hamfinger23","hamfinger23spread","hamfinger2345",
        "hampinch12","hampinch12open","hampinchall",
        "hamcee12","hamceeall","hamceeopen"
    ],
    "C": [
        "hamthumboutmod","hamthumbacrossmod","hamthumbopenmod"
    ],
    "D": [
        "hamdoublebent","hamdoublehooked",
        "hamfingerstraightmod","hamfingerbendmod","hamfingerhookmod"
    ],
    "E": [
        "hamthumb","hamindexfinger","hammiddlefinger",
        "hamringfinger","hampinky","hambetween",
        "hamfingernail","hamfingerpad","hamfingerside","hamfingermidjoint"
    ]
}

# ================= EXTRACT COMMAND ===================
def extract_command_label(label_string):
    try:
        clean = label_string.replace("nan", "'null'")
        d = ast.literal_eval(clean)
        cmd = d.get("Command", "")
        cmd = cmd.replace("\\\\", "\\")
        cmd = cmd.strip().strip("\\")
        return cmd.lower()
    except:
        return ""

# ================= SPLIT NODE COMPONENTS =============
def split_combined_label_to_nodes(cmd):
    tokens = [t.strip() for t in cmd.split("\\") if t.strip() != ""]
    out = {}
    for node, valid in NODES.items():
        out[node] = next((t for t in tokens if t in valid), None)
    return out

# ================== MLP MODEL =========================
def build_mlp(input_dim, n_classes):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.BatchNormalization(),
        layers.Dense(256, activation='relu'), layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(128, activation='relu'), layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# ================= LOAD CSV ==========================
#print("📌 Loading:", DATA_CSV)
df = pd.read_csv(DATA_CSV)

# Extract 63 features
X_all = df[[str(i) for i in range(63)]].values.astype('float32')

# Extract labels (Command)
labels = df['label'].astype(str).apply(extract_command_label).values

# ================= BUILD NODE DATASETS ================
X_node = {n: [] for n in NODES}
y_node = {n: [] for n in NODES}

for i, lab in enumerate(labels):
    mapping = split_combined_label_to_nodes(lab)
    for node in NODES:
        lbl = mapping[node]
        if lbl is not None:
            X_node[node].append(X_all[i])
            y_node[node].append(lbl)

# ================= TRAIN ALL NODES ====================
for node in NODES:
    if len(y_node[node]) == 0:
        #print(f"⚠️ Node {node} has NO samples.")
        continue

    #print(f"\n🔷 Training Node {node} with {len(y_node[node])} samples...")

    Xn = np.array(X_node[node])
    yn = np.array(y_node[node])
    le = LabelEncoder()
    y_enc = le.fit_transform(yn)

    X_train, X_val, y_train, y_val = train_test_split(
        Xn, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

    if __name__ == "__main__":

      model = build_mlp(63, len(le.classes_))

      es = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True
      )

      model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=80,
        batch_size=32,
        callbacks=[es],
        verbose=2
    )

      preds = np.argmax(model.predict(X_val), axis=1)
      acc = accuracy_score(y_val, preds)

      model.save(f"{OUT_DIR}/{node}.h5")
      joblib.dump(le, f"{OUT_DIR}/{node}_encoder.joblib")
    '''

    model = build_mlp(63, len(le.classes_))
    es = callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)




    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=80,
        batch_size=32,
        callbacks=[es],
        verbose=2
    )

    preds = np.argmax(model.predict(X_val), axis=1)
    acc = accuracy_score(y_val, preds)

    #print(f"🎯 Node {node} Accuracy = {acc:.4f}")

    model.save(f"{OUT_DIR}/{node}.h5")
    joblib.dump(le, f"{OUT_DIR}/{node}_encoder.joblib")
    #print(f"💾 Saved {OUT_DIR}/{node}.h5 and encoder")

#print("\n🔥 TRAINING COMPLETE — MODELS SAVED!")
'''


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/handshape_combinations_features_updated.csv'

In [ ]:
# ==========================================================
#   FINAL — STRICT HIERARCHICAL INFERENCE (WITH DAG RULES)
# ==========================================================
import os, joblib
import numpy as np
import tensorflow as tf
import cv2
import mediapipe as mp

MODEL_DIR = "/content/drive/MyDrive/hier_sravs_models"

# ---- DAG: Allowed nodes based on A prediction ----
ALLOWED = {
    "hamcee12": ["D"],
    "hamceeall": ["D"],
    "hamceeopen": ["D"],

    "hamfist": ["C", "D", "E"],
    "hamflathand": ["C", "D", "E"],

    "hampinch12": ["D", "E"],
    "hampinch12open": ["D", "E"],
    "hampinchall": ["D", "E"],

    "hamfinger2": ["D"],
    "hamfinger23": ["D"],
    "hamfinger23spread": ["D"],
    "hamfinger2345": ["D"]
}

mp_hands = mp.solutions.hands

# ==========================================================
#   Convert image → 63 landmark vector
# ==========================================================
def image_to_vector(img_path):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(img_path)

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:
        res = hands.process(rgb)
        if not res.multi_hand_landmarks:
            return None

        lm = res.multi_hand_landmarks[0]
        vec = []
        for p in lm.landmark:
            vec.extend([p.x, p.y, p.z])

        return np.array(vec, dtype="float32").reshape(1, 63)

# ==========================================================
#   Base prediction function for each node
# ==========================================================
def predict_node(node_name, vec):
    model_path = f"{MODEL_DIR}/{node_name}.h5"
    enc_path   = f"{MODEL_DIR}/{node_name}_encoder.joblib"

    if not os.path.exists(model_path):
        return None

    model = tf.keras.models.load_model(model_path)
    le = joblib.load(enc_path)
    probs = model.predict(vec)[0]

    idx = np.argmax(probs)
    return le.inverse_transform([idx])[0]

def predict_node_top3_E(vec):
    node_name = "E"
    model_path = f"{MODEL_DIR}/{node_name}.h5"
    enc_path   = f"{MODEL_DIR}/{node_name}_encoder.joblib"

    if not os.path.exists(model_path):
        return None

    model = tf.keras.models.load_model(model_path)
    le = joblib.load(enc_path)
    probs = model.predict(vec)[0]

    idxs = probs.argsort()[::-1][:3]
    return [(le.inverse_transform([i])[0], float(probs[i])) for i in idxs]

# ==========================================================
#   HIERARCHICAL INFERENCE WITH DAG LOGIC
# ==========================================================
def hierarchical_predict(vec):
    result = {}

    # -------- 1. Predict A ALWAYS --------
    A_pred = predict_node("A", vec)
    result["A"] = A_pred

    # If A prediction not in allowed dict (rare), fallback = predict all
    allowed_nodes = ALLOWED.get(A_pred, ["C", "D", "E"])

    # -------- 2. Predict C (only if allowed) --------
    if "C" in allowed_nodes:
        result["C"] = predict_node("C", vec)
    else:
        result["C"] = None

    # -------- 3. Predict D (only if allowed) --------
    if "D" in allowed_nodes:
        result["D"] = predict_node("D", vec)
    else:
        result["D"] = None

    # -------- 4. Predict E (only if allowed) --------
    if "E" in allowed_nodes:
        result["E"] = predict_node_top3_E(vec)
    else:
        result["E"] = None

    return result

# ==========================================================
#   IMAGE → FULL A,C,D,E prediction
# ==========================================================
def predict_hierarchy_from_image(img_path):
    vec = image_to_vector(img_path)
    if vec is None:
        return {"error": "NO_HAND_DETECTED"}
    return hierarchical_predict(vec)


# ==========================================================
# TEST EXAMPLE
# ==========================================================
test_img = "/content/drive/MyDrive/handshape_combinations/hamcee12,hamthumbopenmod/Handshape/image_001.jpg"
#print("🔍 Predicting:", test_img)

out = predict_hierarchy_from_image(test_img)
#print("\nSTRICT HIERARCHICAL OUTPUT:")
#print(out)


🔍 Predicting: /content/drive/MyDrive/handshape_combinations/hamcee12,hamthumbopenmod/Handshape/image_001.jpg


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step

STRICT HIERARCHICAL OUTPUT:
{'A': 'hamcee12', 'C': None, 'D': 'hamfingerstraightmod', 'E': None}


In [ ]:
# ==========================================================
#   AUTO-DAG DISCOVERY + TRAINING SCRIPT
# ==========================================================
import os, ast, json
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

DATA_CSV = "/content/drive/MyDrive/handshape_combinations_features_updated.csv"
MODEL_DIR = "/content/drive/MyDrive/hier_sravs_models"
DAG_JSON = "/content/drive/MyDrive/hier_sravs_models/dag_rules.json"
os.makedirs(MODEL_DIR, exist_ok=True)

# Node definitions
NODE_A = [
    "hamfist","hamflathand","hamfinger2","hamfinger23","hamfinger23spread","hamfinger2345",
    "hampinch12","hampinch12open","hampinchall","hamcee12","hamceeall","hamceeopen"
]

NODE_C = ["hamthumboutmod","hamthumbacrossmod","hamthumbopenmod"]

NODE_D = ["hamdoublebent","hamdoublehooked","hamfingerstraightmod","hamfingerbendmod","hamfingerhookmod"]

NODE_E = [
    "hamthumb","hamindexfinger","hammiddlefinger","hamringfinger","hampinky",
    "hambetween","hamfingernail","hamfingerpad","hamfingerside","hamfingermidjoint"
]

# ----------------------------------------------------------
#   Clean and extract the command
# ----------------------------------------------------------
def clean_command(s):
    try:
        s = s.replace("nan", "'null'")
        d = ast.literal_eval(s)
        cmd = d["Command"].replace("\\\\", "\\").strip("\\").lower()
        return cmd
    except:
        return ""

def split_nodes(cmd):
    tokens = [t.strip() for t in cmd.split("\\") if t.strip()]
    A = next((t for t in tokens if t in NODE_A), None)
    C = next((t for t in tokens if t in NODE_C), None)
    D = next((t for t in tokens if t in NODE_D), None)
    E = next((t for t in tokens if t in NODE_E), None)
    return A, C, D, E

# ----------------------------------------------------------
#   LOAD CSV + EXTRACT FEATURES + LABELS
# ----------------------------------------------------------
df = pd.read_csv(DATA_CSV)
X_all = df[[str(i) for i in range(63)]].values.astype("float32")

commands = df["label"].astype(str).apply(clean_command)
split_labels = commands.apply(split_nodes)

A_list = []
C_list = []
D_list = []
E_list = []

for a, c, d, e in split_labels:
    A_list.append(a)
    C_list.append(c)
    D_list.append(d)
    E_list.append(e)

# ----------------------------------------------------------
#   AUTO-DISCOVER DAG RULES
# ----------------------------------------------------------
DAG = {}

for i, A in enumerate(A_list):
    if A is None:
        continue

    if A not in DAG:
        DAG[A] = set()

    if C_list[i] is not None:
        DAG[A].add("C")
    if D_list[i] is not None:
        DAG[A].add("D")
    if E_list[i] is not None:
        DAG[A].add("E")

# Convert sets → lists
DAG = {k: sorted(list(v)) for k, v in DAG.items()}

with open(DAG_JSON, "w") as f:
    json.dump(DAG, f, indent=4)

#print("🔥 AUTO-DAG DISCOVERED:")
#print(json.dumps(DAG, indent=4))

# ----------------------------------------------------------
#   PREPARE DATA FOR EACH NODE
# ----------------------------------------------------------
X_node = {"A": [], "C": [], "D": [], "E": []}
y_node = {"A": [], "C": [], "D": [], "E": []}

for i in range(len(A_list)):
    if A_list[i] is None:
        continue

    # A always applies
    X_node["A"].append(X_all[i])
    y_node["A"].append(A_list[i])

    if C_list[i] is not None:
        X_node["C"].append(X_all[i])
        y_node["C"].append(C_list[i])

    if D_list[i] is not None:
        X_node["D"].append(X_all[i])
        y_node["D"].append(D_list[i])

    if E_list[i] is not None:
        X_node["E"].append(X_all[i])
        y_node["E"].append(E_list[i])

# ----------------------------------------------------------
#   MLP BUILDER
# ----------------------------------------------------------
def build_mlp(n_classes):
    model = models.Sequential([
        layers.Input(shape=(63,)),
        layers.BatchNormalization(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# ----------------------------------------------------------
#   TRAIN MODELS FOR EACH NODE
# ----------------------------------------------------------
for node in ["A", "C", "D", "E"]:
    if len(y_node[node]) == 0:
        #print(f"⚠️ Node {node} has NO samples.")
        continue

    Xn = np.array(X_node[node])
    yn = np.array(y_node[node])
    le = LabelEncoder()
    y_enc = le.fit_transform(yn)

    X_train, X_val, y_train, y_val = train_test_split(
        Xn, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

    model = build_mlp(len(le.classes_))
    es = callbacks.EarlyStopping(patience=8, restore_best_weights=True)

    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=80, batch_size=32, callbacks=[es], verbose=2)

    model.save(f"{MODEL_DIR}/{node}.h5")
    joblib.dump(le, f"{MODEL_DIR}/{node}_encoder.joblib")

#print("🎉 TRAINING + AUTO-DAG SAVED!")


🔥 AUTO-DAG DISCOVERED:
{
    "hampinchall": [
        "D"
    ],
    "hampinch12open": [
        "D",
        "E"
    ],
    "hampinch12": [
        "D",
        "E"
    ],
    "hamflathand": [
        "C",
        "D",
        "E"
    ],
    "hamfist": [
        "C",
        "D",
        "E"
    ],
    "hamfinger23spread": [
        "C",
        "D",
        "E"
    ],
    "hamfinger2345": [
        "C",
        "D",
        "E"
    ],
    "hamfinger23": [
        "C",
        "D",
        "E"
    ],
    "hamfinger2": [
        "C",
        "D",
        "E"
    ],
    "hamceeopen": [
        "C",
        "D",
        "E"
    ],
    "hamceeall": [
        "C",
        "D"
    ],
    "hamcee12": [
        "C",
        "D",
        "E"
    ]
}
Epoch 1/80
149/149 - 7s - 49ms/step - accuracy: 0.4103 - loss: 1.6883 - val_accuracy: 0.4193 - val_loss: 1.7432
Epoch 2/80
149/149 - 1s - 7ms/step - accuracy: 0.5945 - loss: 1.0813 - val_accuracy: 0.6849 - val_loss: 1.1495
Epoch 3/80
149/149 - 1s -

Epoch 1/80
99/99 - 2s - 25ms/step - accuracy: 0.6704 - loss: 0.7199 - val_accuracy: 0.7336 - val_loss: 0.7917
Epoch 2/80
99/99 - 0s - 4ms/step - accuracy: 0.8276 - loss: 0.4449 - val_accuracy: 0.7904 - val_loss: 0.5449
Epoch 3/80
99/99 - 0s - 4ms/step - accuracy: 0.8626 - loss: 0.3343 - val_accuracy: 0.8220 - val_loss: 0.4076
Epoch 4/80
99/99 - 0s - 4ms/step - accuracy: 0.8800 - loss: 0.2936 - val_accuracy: 0.8838 - val_loss: 0.3009
Epoch 5/80
99/99 - 0s - 4ms/step - accuracy: 0.8986 - loss: 0.2600 - val_accuracy: 0.9129 - val_loss: 0.2188
Epoch 6/80
99/99 - 0s - 4ms/step - accuracy: 0.8971 - loss: 0.2513 - val_accuracy: 0.9419 - val_loss: 0.1707
Epoch 7/80
99/99 - 0s - 4ms/step - accuracy: 0.9198 - loss: 0.2075 - val_accuracy: 0.9444 - val_loss: 0.1450
Epoch 8/80
99/99 - 0s - 4ms/step - accuracy: 0.9189 - loss: 0.2056 - val_accuracy: 0.9545 - val_loss: 0.1229
Epoch 9/80
99/99 - 0s - 4ms/step - accuracy: 0.9261 - loss: 0.1901 - val_accuracy: 0.9583 - val_loss: 0.1328
Epoch 10/80
99/99 

Epoch 1/80
101/101 - 2s - 23ms/step - accuracy: 0.5855 - loss: 1.0463 - val_accuracy: 0.6372 - val_loss: 1.2375
Epoch 2/80
101/101 - 1s - 7ms/step - accuracy: 0.6921 - loss: 0.8097 - val_accuracy: 0.7132 - val_loss: 1.0031
Epoch 3/80
101/101 - 1s - 7ms/step - accuracy: 0.7420 - loss: 0.6808 - val_accuracy: 0.7943 - val_loss: 0.7357
Epoch 4/80
101/101 - 1s - 7ms/step - accuracy: 0.7667 - loss: 0.5919 - val_accuracy: 0.8429 - val_loss: 0.5238
Epoch 5/80
101/101 - 1s - 7ms/step - accuracy: 0.7885 - loss: 0.5277 - val_accuracy: 0.8741 - val_loss: 0.4019
Epoch 6/80
101/101 - 1s - 9ms/step - accuracy: 0.8169 - loss: 0.4830 - val_accuracy: 0.8591 - val_loss: 0.3613
Epoch 7/80
101/101 - 1s - 9ms/step - accuracy: 0.8200 - loss: 0.4491 - val_accuracy: 0.8815 - val_loss: 0.3259
Epoch 8/80
101/101 - 1s - 10ms/step - accuracy: 0.8281 - loss: 0.4380 - val_accuracy: 0.8890 - val_loss: 0.3007
Epoch 9/80
101/101 - 1s - 8ms/step - accuracy: 0.8475 - loss: 0.3842 - val_accuracy: 0.9040 - val_loss: 0.2731

Epoch 1/80
49/49 - 4s - 79ms/step - accuracy: 0.4316 - loss: 1.5178 - val_accuracy: 0.3632 - val_loss: 1.7301
Epoch 2/80
49/49 - 0s - 6ms/step - accuracy: 0.5895 - loss: 1.0884 - val_accuracy: 0.3555 - val_loss: 1.6527
Epoch 3/80
49/49 - 0s - 5ms/step - accuracy: 0.6566 - loss: 0.8838 - val_accuracy: 0.5064 - val_loss: 1.4000
Epoch 4/80
49/49 - 0s - 5ms/step - accuracy: 0.7391 - loss: 0.7227 - val_accuracy: 0.6777 - val_loss: 1.1036
Epoch 5/80
49/49 - 0s - 5ms/step - accuracy: 0.7871 - loss: 0.5956 - val_accuracy: 0.8286 - val_loss: 0.8790
Epoch 6/80
49/49 - 0s - 5ms/step - accuracy: 0.8095 - loss: 0.5327 - val_accuracy: 0.8517 - val_loss: 0.6487
Epoch 7/80
49/49 - 0s - 5ms/step - accuracy: 0.8376 - loss: 0.4369 - val_accuracy: 0.9028 - val_loss: 0.4666
Epoch 8/80
49/49 - 0s - 5ms/step - accuracy: 0.8421 - loss: 0.4349 - val_accuracy: 0.9028 - val_loss: 0.3832
Epoch 9/80
49/49 - 0s - 5ms/step - accuracy: 0.8606 - loss: 0.3728 - val_accuracy: 0.9463 - val_loss: 0.2674
Epoch 10/80
49/49 

🎉 TRAINING + AUTO-DAG SAVED!


In [ ]:
# ==========================================================
#   FINAL INFERENCE USING AUTO-DAG
# ==========================================================
import os, json, joblib
import numpy as np
import tensorflow as tf
import cv2
import mediapipe as mp

MODEL_DIR = "/content/drive/MyDrive/hier_sravs_models"
DAG_JSON = "/content/drive/MyDrive/hier_sravs_models/dag_rules.json"

DAG = json.load(open(DAG_JSON))

mp_hands = mp.solutions.hands

def image_to_vector(img_path):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(img_path)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with mp_hands.Hands(static_image_mode=True, max_num_hands=1) as hands:
        res = hands.process(rgb)
        if not res.multi_hand_landmarks:
            return None
        lm = res.multi_hand_landmarks[0]
        vec = []
        for p in lm.landmark:
            vec.extend([p.x, p.y, p.z])
        return np.array(vec, dtype="float32").reshape(1, 63)

def predict_node(node, vec):
    path = f"{MODEL_DIR}/{node}.h5"
    if not os.path.exists(path):
        return None
    model = tf.keras.models.load_model(path)
    le = joblib.load(f"{MODEL_DIR}/{node}_encoder.joblib")
    probs = model.predict(vec)[0]
    return le.inverse_transform([np.argmax(probs)])[0]

def predict_node_top3_E(vec):
    path = f"{MODEL_DIR}/E.h5"
    if not os.path.exists(path):
        return None
    model = tf.keras.models.load_model(path)
    le = joblib.load(f"{MODEL_DIR}/E_encoder.joblib")
    probs = model.predict(vec)[0]
    idxs = probs.argsort()[::-1][:3]
    return [(le.inverse_transform([i])[0], float(probs[i])) for i in idxs]

def hierarchical_predict(img_path):
    vec = image_to_vector(img_path)
    if vec is None:
        return {"error": "No hand detected"}

    # 1. Predict A
    A = predict_node("A", vec)
    allowed = DAG.get(A, [])

    result = {"A": A}

    # 2. Predict C / D / E based on DAG
    result["C"] = predict_node("C", vec) if "C" in allowed else None
    result["D"] = predict_node("D", vec) if "D" in allowed else None
    result["E"] = predict_node_top3_E(vec) if "E" in allowed else None

    return result

# Test image
test_img = "/content/drive/MyDrive/handshape_combinations/hamcee12,hamthumbopenmod/Handshape/image_001.jpg"
#print("OUTPUT:", hierarchical_predict(test_img))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 418ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
OUTPUT: {'A': 'hamcee12', 'C': 'hamthumbopenmod', 'D': 'hamfingerstraightmod', 'E': [('hammiddlefinger', 0.9271529316902161), ('hamindexfinger', 0.07284239679574966), ('hambetween', 4.281566361896694e-06)]}


In [ ]:
import json
DAG = json.load(open("/content/drive/MyDrive/hier_sravs_models/dag_rules.json"))
#print(json.dumps(DAG, indent=4))


{
    "hampinchall": [
        "D"
    ],
    "hampinch12open": [
        "D",
        "E"
    ],
    "hampinch12": [
        "D",
        "E"
    ],
    "hamflathand": [
        "C",
        "D",
        "E"
    ],
    "hamfist": [
        "C",
        "D",
        "E"
    ],
    "hamfinger23spread": [
        "C",
        "D",
        "E"
    ],
    "hamfinger2345": [
        "C",
        "D",
        "E"
    ],
    "hamfinger23": [
        "C",
        "D",
        "E"
    ],
    "hamfinger2": [
        "C",
        "D",
        "E"
    ],
    "hamceeopen": [
        "C",
        "D",
        "E"
    ],
    "hamceeall": [
        "C",
        "D"
    ],
    "hamcee12": [
        "C",
        "D",
        "E"
    ]
}


In [11]:
def infer_handshape_from_frame(frame):
    """
    Convert frame → 63D vector → hierarchical prediction
    """

    import tempfile
    import os

    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        temp_path = tmp.name
        cv2.imwrite(temp_path, frame)

    try:
        vec = image_to_vector(temp_path)

        if vec is None:
            return None

        result = hierarchical_predict(temp_path)

        # return ONLY Node A (main handshape)
        return result.get("A")

    finally:
        os.remove(temp_path)

In [12]:
from collections import Counter
import cv2

def run_handshape_module(video_path):

    predictions = []

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None   # integration expects silent failure

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        pred = infer_handshape_from_frame(frame)

        if pred is not None:
            predictions.append(pred)

    cap.release()

    if len(predictions) == 0:
        return None

    final_label = Counter(predictions).most_common(1)[0][0]

    return final_label   # ✅ ONLY FINAL OUTPUT